# 22 | LangChain Middleware：对话太长，先总结再继续聊

产品经理和需求梳理 Agent 聊一个“订单报表导出”功能。

前面已经聊了导出字段、权限规则、文件格式、失败处理、上线范围。聊到后面，产品经理问：

> 把前面定下来的内容整理成研发评审要点。

如果把所有历史消息都塞给模型，两个问题会同时出现：上下文窗口吃紧，调用成本也跟着涨。

`SummarizationMiddleware` 解决的不是“写一篇摘要”，而是：**旧对话压缩成摘要，最近对话继续保留，让 Agent 还能接着干活。**

## 一、为什么不能无限塞历史消息

长对话里不是每句话都同等重要。

比如需求梳理场景里，真正关键的是：功能目标、字段范围、权限规则、技术约束、已经定下来的取舍。

至于中间那些“这个先不做”“刚才那条改一下”“我再补充一个点”，全塞给模型只会占上下文，也会增加调用成本。

`SummarizationMiddleware` 的思路很简单：

- 未达到触发条件：历史消息原样保留
- 达到触发条件：旧消息总结成一段摘要
- 最近几轮：继续保留原文，避免丢掉最新细节

## 二、准备模型和观察函数

下面会构造一段较长的需求梳理对话，并打印模型真正看到的消息。

这里分两个模型：主模型负责最终回答；总结模型只负责压缩旧对话，所以可以用本地模型。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import AgentState, SummarizationMiddleware, before_model
from langchain.chat_models import init_chat_model

load_dotenv(dotenv_path=".env")

main_model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)

# 总结模型可以和主模型不同。
# 这里用本地 Ollama 模型做摘要，减少旧对话压缩这一步的云端调用成本。
summary_model = init_chat_model("ollama:qwen2.5:14b")


@before_model
def show_model_context(state: AgentState, runtime):
    """打印模型调用前实际保留下来的上下文。"""
    print(f"[模型实际看到的消息数量] {len(state['messages'])}")
    for index, message in enumerate(state["messages"], start=1):
        # 不截断 summary，方便确认旧对话细节到底有没有被保留下来。
        print(f"\n{index}. {message.__class__.__name__}")
        print(message.content)
    return None

## 三、触发自动总结

为了让效果明显，这里用 `trigger=("messages", 8)`：消息数量达到 8 条就触发总结。

`keep=("messages", 4)` 表示：旧消息被总结，但最近 4 条消息保留原文。

In [ ]:
requirement_history = [
    {
        "role": "user",
        "content": "我们要做一个订单报表导出功能，主要给运营和财务用。",
    },
    {
        "role": "assistant",
        "content": "已记录：功能是订单报表导出，主要使用人是运营和财务。",
    },
    {
        "role": "user",
        "content": "导出的字段先包括订单号、客户名称、下单时间、支付金额、支付状态。",
    },
    {
        "role": "assistant",
        "content": "已记录字段范围：订单号、客户名称、下单时间、支付金额、支付状态。",
    },
    {
        "role": "user",
        "content": "权限上，普通运营只能导出自己负责区域的数据，财务可以导出全部数据。",
    },
    {
        "role": "assistant",
        "content": "已记录权限规则：运营按区域限制，财务可导出全部数据。",
    },
    {
        "role": "user",
        "content": "导出格式先支持 Excel，不做 PDF。文件太大时可以异步生成。",
    },
    {
        "role": "assistant",
        "content": "已记录导出格式：优先支持 Excel；大文件走异步生成。",
    },
    {
        "role": "user",
        "content": "如果导出失败，要给用户明确提示，并允许重新发起导出。",
    },
    {
        "role": "assistant",
        "content": "已记录异常处理：失败时展示提示，并允许用户重新导出。",
    },
    {
        "role": "user",
        "content": "这个需求优先级比较高，先做后台管理端，移动端暂时不做。",
    },
    {
        "role": "assistant",
        "content": "已记录优先级和范围：优先做后台管理端，移动端暂不支持。",
    },
    {
        "role": "user",
        "content": "现在把前面定下来的内容整理成研发评审要点。",
    },
]

summary_agent = create_agent(
    model=main_model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=summary_model,
            # 达到 8 条消息就触发总结，方便在示例里看到效果。
            trigger=("messages", 8),
            # 最近 4 条消息保留原文，旧消息压缩成摘要。
            keep=("messages", 4),
        ),
        # 放在总结 middleware 后面，用来观察总结后的上下文。
        show_model_context,
    ],
    system_prompt=(
        "你是需求梳理 Agent。请只根据现有上下文整理研发评审要点。"
        "没有明确提到的接口路径、数据阈值、技术方案，不要自行编造；"
        "如果确实需要，可以放到‘待确认问题’里。"
    ),
)

result = summary_agent.invoke({"messages": requirement_history})

print("\n[最终回复]")
print(result["messages"][-1].content)

运行后，模型看到的上下文通常会变成上面那样，第一条是旧对话的 summary，后面是最近几条原始消息。

这里要注意：总结不是无损压缩。它能保留主要信息，但不保证 100% 留住每个细节。实际总结质量取决于总结模型和 `summary_prompt`。

所以，像“不做 PDF”“移动端暂不做”“不能导出全部数据”这类关键约束，如果业务上很重要，就要在后面的 `summary_prompt` 里明确要求保留。

## 四、SummarizationMiddleware 的配置分类

`SummarizationMiddleware` 主要看四件事：

1. 什么时候触发总结：`trigger`
2. 总结后保留多少最近上下文：`keep`
3. 用什么模型做总结：`model`
4. 总结时重点保留什么：`summary_prompt`

这四个配置决定了它到底是“省钱好用”，还是“省了钱但丢了关键信息”。

### 4.1 按 trigger 分类：什么时候总结

`trigger` 决定什么时候开始总结。

| 写法 | 含义 | 适合场景 |
|---|---|---|
| `trigger=("messages", n)` | 消息数量达到 n 时触发 | 演示、调试，最直观 |
| `trigger=("tokens", n)` | token 数达到 n 时触发 | 更接近生产 |
| `trigger=("fraction", x)` | 上下文窗口使用比例达到 x 时触发 | 根据模型窗口自动调整 |

如果用 `tokens`，LangChain 会通过 `token_counter` 估算当前消息占用的 token 数。默认使用近似 token 计数，不是等模型上下文爆了才处理。

如果用 `fraction`，模型需要知道自己的上下文窗口大小。没有模型 profile 时，就要显式提供 `profile={"max_input_tokens": ...}`。

In [ ]:
# 方式一：按消息数量触发。
# 达到 8 条消息就总结，适合演示，读者一眼能看懂。
messages_summary_middleware = SummarizationMiddleware(
    model=summary_model,
    trigger=("messages", 8),
    keep=("messages", 4),
)

# 方式二：按 token 数触发。
# 更接近生产，因为模型上下文窗口最终看的就是 token。
tokens_summary_middleware = SummarizationMiddleware(
    model=summary_model,
    trigger=("tokens", 3000),
    keep=("tokens", 1000),
)

# 方式三：按上下文窗口比例触发。
# 这里给本地模型补充 max_input_tokens，middleware 才知道 80% 是多少 token。
fraction_summary_model = init_chat_model(
    "ollama:qwen2.5:14b",
    profile={"max_input_tokens": 8192},
)

fraction_summary_middleware = SummarizationMiddleware(
    model=fraction_summary_model,
    trigger=("fraction", 0.8),  # 接近上下文窗口 80% 时触发总结。
    keep=("fraction", 0.25),    # 总结后保留最近约 25% 的上下文。
)

### 4.2 按 keep 分类：总结后保留什么

`keep` 决定总结后保留多少最近上下文。

| 写法 | 含义 | 适合场景 |
|---|---|---|
| `keep=("messages", n)` | 保留最近 n 条消息 | 聊天轮次清晰的场景 |
| `keep=("tokens", n)` | 保留最近约 n 个 token | 更精确控制上下文成本 |
| `keep=("fraction", x)` | 保留上下文窗口的 x 比例 | 想跟随模型窗口自动调整 |

可以把 `trigger` 和 `keep` 分开理解：

- `trigger`：什么时候开始压缩旧历史
- `keep`：压缩后，最近多少内容继续保留原文

这也是为什么第三节里 summary 后还有最近几条消息。旧消息被压缩了，但最新决策没有被压掉。

### 4.3 按总结模型分类：谁来写 summary

总结模型可以和主模型相同，也可以不同。

| 方式 | 优点 | 代价 |
|---|---|---|
| 主模型自己总结 | 简单，质量通常稳定 | 成本可能高 |
| 更便宜的小模型总结 | 降低成本 | 需要测试总结质量 |
| 本地模型总结 | 减少云端调用，也适合本地实验 | 依赖本地模型能力和服务稳定性 |

本文用的是本地 `ollama:qwen2.5:14b` 做总结模型，主模型只负责最终回答。这个设计很常见：便宜模型压缩旧历史，强模型处理最终任务。

### 4.4 按 summary_prompt 分类：总结时重点保留什么

默认 `summary_prompt` 已经能把旧对话整理成 `SESSION INTENT`、`SUMMARY`、`NEXT STEPS` 这类结构。

但总结不是无损压缩。像“不做 PDF”“移动端暂不做”“不能导出全部数据”这种否定约束，业务上很重要，却可能被总结模型省略。

如果这些信息很关键，就应该定制 `summary_prompt`。

In [ ]:
custom_summary_prompt = """
请总结下面的历史对话，保留对后续任务有用的信息。

重点保留：
1. 已确认的需求
2. 明确不做或暂不支持的内容
3. 权限规则
4. 格式、范围、异常处理等约束
5. 待确认问题

不要添加历史对话中没有出现的新方案。

历史对话：
{messages}
"""

custom_summary_agent = create_agent(
    model=main_model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=summary_model,
            trigger=("messages", 8),
            keep=("messages", 4),
            # 用自定义 prompt 控制 summary 重点，减少关键信息被压掉。
            summary_prompt=custom_summary_prompt,
        )
    ],
    system_prompt="你是需求梳理 Agent。",
)

## 五、小结

`SummarizationMiddleware` 的重点不是摘要写得多漂亮，而是让 Agent 在长对话里继续工作。

可以这样记：

| 问题 | 处理方式 |
|---|---|
| 历史消息太多 | 用 `trigger` 触发自动总结 |
| 最新细节不能丢 | 用 `keep` 保留最近上下文 |
| 想控制成本 | 总结模型用本地或更便宜的小模型 |
| 想按消息数量演示 | `trigger=("messages", n)` |
| 想按 token 控制成本 | `trigger=("tokens", n)` |
| 想按窗口比例控制 | `trigger=("fraction", x)`，并确保模型有 `max_input_tokens` profile |
| 担心摘要丢约束 | 定制 `summary_prompt` |

下一篇继续看上下文治理：工具结果越来越多时，哪些该留，哪些该清。

参考：

- LangChain Prebuilt Middleware：https://docs.langchain.com/oss/python/langchain/middleware/built-in
- LangChain SummarizationMiddleware API Reference：https://reference.langchain.com/python/langchain/agents/middleware/summarization/SummarizationMiddleware